# Experiment 5: Decision Tree and Random Forest Classification
This standalone notebook implements Decision Tree depth pruning and Random Forest ensemble classification on the Wisconsin Diagnostic Breast Cancer dataset.

In [1]:
import os
import matplotlib
matplotlib.use('Agg') # Strictly headless - non-interfering, zero GUI popups

def resolve_path(rel_path):
    """Dynamically resolves datasets whether run from repo root or Ex subfolder."""
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    """Avoids nested directories if running from within Ex5."""
    if os.path.basename(os.getcwd()) == 'Ex5':
        if rel_path.startswith('Ex5/'):
            return rel_path[len('Ex5/'):]
    return rel_path

import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(resolve_out('Ex5'), exist_ok=True)

In [2]:
def run_experiment_5(csv_path="Datasets/Breast_Cancer/breast_cancer.csv"):
    print("="*60)
    print("=== LAUNCHING EXPERIMENT 5: DECISION TREES & RANDOM FORESTS ===")
    print("="*60)
    
    path = resolve_path(csv_path)
    df = pd.read_csv(path)
    
    if 'id' in df.columns:
        df = df.drop(columns=['id'])
    if 'Unnamed: 32' in df.columns:
        df = df.drop(columns=['Unnamed: 32'])
        
    target_col = 'diagnosis' if 'diagnosis' in df.columns else df.columns[0]
    X = df.drop(columns=[target_col])
    y = df[target_col].map({'M': 1, 'B': 0}) if df[target_col].dtype == 'object' else df[target_col]
    
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    dt = DecisionTreeClassifier(random_state=42)
    dt_grid = GridSearchCV(dt, {'max_depth': [3, 5, 7, None], 'min_samples_split': [2, 5, 10]}, cv=5, scoring='accuracy')
    dt_grid.fit(X_tr, y_tr)
    best_dt = dt_grid.best_estimator_
    
    rf = RandomForestClassifier(random_state=42)
    rf_grid = GridSearchCV(rf, {'n_estimators': [50, 100], 'max_depth': [3, 5, None]}, cv=5, scoring='accuracy')
    rf_grid.fit(X_tr, y_tr)
    best_rf = rf_grid.best_estimator_
    
    results = {}
    for name, model in [('Tuned Decision Tree', best_dt), ('Tuned Random Forest', best_rf)]:
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]
        results[name] = {
            'Accuracy': round(accuracy_score(y_te, y_pred) * 100, 2),
            'Precision': round(precision_score(y_te, y_pred, zero_division=0) * 100, 2),
            'Recall': round(recall_score(y_te, y_pred, zero_division=0) * 100, 2),
            'F1-Score': round(f1_score(y_te, y_pred, zero_division=0) * 100, 2),
            'ROC AUC': round(roc_auc_score(y_te, y_prob), 4)
        }
        
    print("\n=== EXPERIMENT 5 PIPELINE COMPLETE ===")
    return results

In [3]:
# Master Execution Cell
ex5_output = run_experiment_5()
display(pd.DataFrame(ex5_output).T.style.background_gradient(cmap='YlGn', subset=['Accuracy', 'F1-Score']))

=== LAUNCHING EXPERIMENT 5: DECISION TREES & RANDOM FORESTS ===



=== EXPERIMENT 5 PIPELINE COMPLETE ===


,Accuracy,Precision,Recall,F1-Score,ROC AUC
Tuned Decision Tree,92.110000,95.650000,91.670000,93.620000,0.916300
Tuned Random Forest,95.610000,95.890000,97.220000,96.550000,0.993700
